# RTE sweep A: fixing distillation (DeBERTa-v3-xsmall)

The first distilled student (76.8) trailed the plain student (77.7): the teacher's soft labels lean to entailment and are confidently wrong on 17% of train. Two fixes, 5 seeds each, everything else identical to the plain student.

**Unattended:** Runtime → Change runtime type → **GPU (T4)**, then **Runtime → Run all**. Nothing to upload, no Google Drive.
Code is cloned from the public repo; data is rebuilt from public GLUE RTE (fingerprint-checked against the official file);
teacher logits ship in `artifacts/teacher/`. Expected time on a T4: **about 2–2.5 hours**.

Results are downloaded by the browser as soon as each run finishes (a small zip per run, plus `all_results.zip` at the end),
so **keep this tab open**, and allow automatic downloads for colab.research.google.com in Chrome beforehand
(chrome://settings/content/automaticDownloads), or Chrome will pause on a "download multiple files" prompt.
Checkpoints are not downloaded; every number is in the logs and zips.

## Setup

In [ ]:
!nvidia-smi --query-gpu=name,memory.total --format=csv
import os, sys
%cd /content
if os.path.isdir("/content/rte-challenge/.git"):
    !git -C /content/rte-challenge pull -q --ff-only
else:
    !git clone -q https://github.com/ANONYMIZED/rte-challenge.git
%cd /content/rte-challenge
!git log --oneline -1
!grep -vE '^(torch|nvidia-|triton|cuda-)' requirements.txt > /tmp/req-colab.txt
!pip install -q -r /tmp/req-colab.txt
!pip install -q --no-deps -e .
# train.jsonl is rebuilt from public GLUE RTE and checked against the official file's fingerprint
!python scripts/prepare_data.py --out data

name, memory.total [MiB]
Tesla T4, 15360 MiB
/content
/content/rte-challenge
a869b48 (HEAD -> setup/project-scaffold, origin/setup/project-scaffold, origin/main, origin/HEAD) Add unattended Colab sweeps: KD fixes, factorised embedding, DeBERTa-v3-small
ERROR: pip's dependency resolver does not currently take into account all the packages that are installed. This behaviour is the source of the following dependency conflicts.
ipython 7.34.0 requires jedi>=0.16, which is not installed.
google-colab 1.0.0 requires pandas==2.2.3, but you have pandas 3.1.1 which is incompatible.
gradio 6.31.0 requires pillow<13.0,>=8.0, but you have pillow 13.0.0 which is incompatible.
betterproto 2.0.0b7 requires python-dateutil<3.0,>=2.8, but you have python-dateutil 4.0.0 which is incompatible.
dask-cudf-cu13 26.6.0 requires pandas<2.4.0,>=2.0, but you have pandas 3.1.1 which is incompatible.
gcsfs 2025.12.0 requires fsspec==2025.12.0, but you have fsspec 2026.7.0 which is incompatible.
bigframes 2.51.0 r

In [ ]:
import json, os, subprocess
from google.colab import files

def run(cmd: str, run_name: str) -> None:
    """Train, stream the log, then download outputs/<run_name> (no checkpoint) as a small zip.
    A failure is reported and the notebook moves on to the next run."""
    print(f">> {cmd}", flush=True)
    env = {**os.environ, "PYTHONUNBUFFERED": "1"}
    p = subprocess.Popen(cmd, shell=True, stdout=subprocess.PIPE, stderr=subprocess.STDOUT,
                         text=True, bufsize=1, env=env)
    for line in p.stdout:
        print(line, end="")
    if p.wait() != 0:
        print(f"!! {run_name} failed (exit {p.returncode}); continuing", flush=True)
    m = f"outputs/{run_name}/metrics.json"
    if os.path.exists(m):
        r = json.load(open(m))
        print(f"RESULT {run_name}: dev {r['dev_acc_mean']*100:.1f} ± {r['dev_acc_std']*100:.1f}, "
              f"params {r['params']['total']/1e6:.1f}M", flush=True)
    if os.path.isdir(f"outputs/{run_name}"):
        z = f"{run_name}.zip"
        subprocess.run(f"zip -q -r {z} outputs/{run_name} -x 'outputs/{run_name}/model/*'", shell=True)
        files.download(z)

## A1. Masked KD: gold-label CE only where the teacher is wrong (alpha 0.5, T 2)

In [ ]:
run("python -m rte.distill --config configs/distill_masked_deberta_v3_xsmall.yaml", "distill_masked_deberta_v3_xsmall")

>> python -m rte.distill --config configs/distill_masked_deberta_v3_xsmall.yaml
/usr/local/lib/python3.13/dist-packages/pandas/core/arrays/masked.py:63: UserWarning: Pandas requires version '1.5.0' or newer of 'bottleneck' (version '1.4.2' currently installed).
  from pandas.core import (
run distill_masked_deberta_v3_xsmall | model microsoft/deberta-v3-xsmall | device cuda
/usr/local/lib/python3.13/dist-packages/torch/jit/_script.py:1491: FutureWarning: `torch.jit.script` is deprecated. Please switch to `torch.compile` or `torch.export`.
  warnings.warn(

Loading weights: 100%|██████████| 198/198 [00:00<00:00, 1364.30it/s]
[transformers] DebertaV2ForSequenceClassification LOAD REPORT from: microsoft/deberta-v3-xsmall
Key                                        | Status     | 
-------------------------------------------+------------+-
lm_predictions.lm_head.dense.bias          | UNEXPECTED | 
lm_predictions.lm_head.bias                | UNEXPECTED | 
mask_predictions.dense.weight       

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

## A2. Re-weighted KD: alpha 0.8, T 4

In [ ]:
run("python -m rte.distill --config configs/distill_a08_t4_deberta_v3_xsmall.yaml", "distill_a08_t4_deberta_v3_xsmall")

>> python -m rte.distill --config configs/distill_a08_t4_deberta_v3_xsmall.yaml
/usr/local/lib/python3.13/dist-packages/pandas/core/arrays/masked.py:63: UserWarning: Pandas requires version '1.5.0' or newer of 'bottleneck' (version '1.4.2' currently installed).
  from pandas.core import (
run distill_a08_t4_deberta_v3_xsmall | model microsoft/deberta-v3-xsmall | device cuda
/usr/local/lib/python3.13/dist-packages/torch/jit/_script.py:1491: FutureWarning: `torch.jit.script` is deprecated. Please switch to `torch.compile` or `torch.export`.
  warnings.warn(

Loading weights: 100%|██████████| 198/198 [00:00<00:00, 1845.28it/s]
[transformers] DebertaV2ForSequenceClassification LOAD REPORT from: microsoft/deberta-v3-xsmall
Key                                        | Status     | 
-------------------------------------------+------------+-
mask_predictions.classifier.weight         | UNEXPECTED | 
deberta.embeddings.word_embeddings._weight | UNEXPECTED | 
lm_predictions.lm_head.LayerNorm.bia

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

## Summary and download everything

In [ ]:
import glob
for f in sorted(glob.glob("outputs/*/metrics.json")):
    r = json.load(open(f)); p = r["params"]
    print(f"{r['run_name']:36s} dev {r['dev_acc_mean']*100:5.1f} ± {r['dev_acc_std']*100:4.1f}  "
          f"val {sum(s['best_val_acc'] for s in r['seeds'])/len(r['seeds'])*100:5.1f}  "
          f"params {p['total']/1e6:6.1f}M (emb {p['word_embedding']/1e6:5.1f}M)")
!zip -q -r all_results.zip outputs -x 'outputs/*/model/*'
files.download("all_results.zip")

distill_a08_t4_deberta_v3_xsmall     dev  77.3 ±  0.8  val  79.5  params   70.8M (emb  49.2M)
distill_masked_deberta_v3_xsmall     dev  78.3 ±  1.4  val  79.2  params   70.8M (emb  49.2M)


<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

In [ ]:
!nvidia-smi --query-gpu=name --format=csv,noheader
%cd /content
!rm -rf rte-challenge && git clone -q https://github.com/ANONYMIZED/rte-challenge.git
%cd /content/rte-challenge
!git log --oneline -1
!grep -vE '^(torch|nvidia-|triton|cuda-)' requirements.txt > /tmp/req-colab.txt
!pip install -q -r /tmp/req-colab.txt
!pip install -q --no-deps -e .
!python scripts/prepare_data.py --out data
# same config and seed as the sweep: expect "seed 21: val 0.8270 dev 0.7798"
!python -m rte.distill --config configs/distill_masked_deberta_v3_xsmall.yaml --seeds 21
!python -m rte.prune --checkpoint outputs/distill_masked_deberta_v3_xsmall --out outputs/distill_masked_deberta_v3_xsmall_pruned
!python scripts/export_predictions.py --run outputs/distill_masked_deberta_v3_xsmall_pruned
!zip -q -r submission_run.zip outputs submission -x 'outputs/*/model/*'
!zip -q -r pruned_model.zip outputs/distill_masked_deberta_v3_xsmall_pruned/model
from google.colab import files
files.download("submission_run.zip")   # small: metrics, prune.json, predictions
files.download("pruned_model.zip")     # ~180 MB, optional

Tesla T4
/content
/content/rte-challenge
89fea8a (HEAD -> setup/project-scaffold, origin/setup/project-scaffold, origin/main, origin/HEAD) Add report, model card, dev predictions and seminar page
ERROR: pip's dependency resolver does not currently take into account all the packages that are installed. This behaviour is the source of the following dependency conflicts.
ipython 7.34.0 requires jedi>=0.16, which is not installed.
google-colab 1.0.0 requires pandas==2.2.3, but you have pandas 3.1.1 which is incompatible.
numba 0.64.0 requires numpy<2.5,>=1.22, but you have numpy 2.5.4 which is incompatible.
gcsfs 2025.12.0 requires fsspec==2025.12.0, but you have fsspec 2026.7.0 which is incompatible.
gradio 6.31.0 requires pillow<13.0,>=8.0, but you have pillow 13.0.0 which is incompatible.
holidays 0.108 requires python-dateutil<3,>=2.9.0.post0, but you have python-dateutil 4.0.0 which is incompatible.
cudf-cu13 26.6.0 requires pandas<2.4.0,>=2.0, but you have pandas 3.1.1 which is incom

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>